In [1]:
import numpy as np
import pandas as pd
from scipy.special import gamma
import requests
import os
import mpmath as mp

# Configurar precisão MAIOR para cálculos com mpmath
mp.dps = 50  # 50 dígitos decimais de precisão (aumentado de 15)

class RiemannZetaFeatureGenerator:
    """
    Classe para gerar features relacionadas aos zeros da função Zeta de Riemann
    baseada no artigo "Predicting Zeros of the Riemann Zeta Function Using Machine Learning"
    """
    
    def __init__(self, n_zeros=100000, start_index=1001):
        """
        Inicializa o gerador de features
        
        Args:
            n_zeros: número total de zeros a considerar
            start_index: índice inicial (para evitar edge effects)
        """
        self.n_zeros = n_zeros
        self.start_index = start_index
        self.gram_points = None
        self.co_gram_points = None
        self.zeros = None
        self.distance = None
        
    def compute_theta(self, t):
        """
        Calcula a função theta de Riemann-Siegel
        θ(t) = arg[Γ(1/4 + it/2)] - (ln π / 2) * t
        
        Usando expansão assintótica (equação 22 do artigo):
        θ(t) = t/2 * ln(t/2π) - t/2 - π/8 + 1/(48t) + 7/(5760t³) + ...
        """
        if t <= 0:
            return 0
        
        # Usar mpmath para maior precisão
        t_mp = mp.mpf(t)
        theta = (t_mp/2) * mp.log(t_mp/(2*mp.pi)) - t_mp/2 - mp.pi/8
        theta += 1/(48*t_mp)
        theta += 7/(5760 * t_mp**3)
        
        return float(theta)
    
    def compute_gram_point(self, n, max_iter=100, tol=1e-10):
        """
        Calcula o n-ésimo ponto de Gram
        Gram point g_n é a solução de θ(g_n) = (n-1)π
        
        Args:
            n: índice do ponto de Gram
            max_iter: número máximo de iterações para Newton-Raphson
            tol: tolerância para convergência
        """
        target = (n - 1) * np.pi
        
        # Estimativa inicial usando aproximação assintótica
        t = 2 * np.pi * np.exp(1 + target / (2 * n))
        
        # Método de Newton-Raphson
        for _ in range(max_iter):
            theta_t = self.compute_theta(t)
            error = theta_t - target
            
            if abs(error) < tol:
                break
            
            # Derivada de theta (aproximada)
            dt = 0.0001
            theta_derivative = (self.compute_theta(t + dt) - theta_t) / dt
            
            if abs(theta_derivative) > 1e-10:
                t = t - error / theta_derivative
        
        return t
    
    def compute_co_gram_point(self, n, max_iter=100, tol=1e-10):
        """
        Calcula o n-ésimo co-Gram point
        Co-Gram point é a solução de θ(t) = nπ + π/2
        """
        target = n * np.pi + np.pi / 2
        
        # Estimativa inicial
        t = 2 * np.pi * np.exp(1 + target / (2 * n))
        
        # Método de Newton-Raphson
        for _ in range(max_iter):
            theta_t = self.compute_theta(t)
            error = theta_t - target
            
            if abs(error) < tol:
                break
            
            dt = 0.0001
            theta_derivative = (self.compute_theta(t + dt) - theta_t) / dt
            
            if abs(theta_derivative) > 1e-10:
                t = t - error / theta_derivative
        
        return t
    
    def compute_Z_function(self, t):
        """
        Calcula a função Z de Riemann-Siegel em t
        Z(t) = exp(iθ(t)) * ζ(1/2 + it)
        
        Para t real, Z(t) é real e Z(t) = |ζ(1/2 + it)| * cos(θ(t) - arg(ζ))
        
        Implementação melhorada com tratamento de erros
        """
        try:
            # Converter para mpmath com alta precisão
            t_mp = mp.mpf(t)
            
            # Calcular s = 1/2 + it
            s = mp.mpc(0.5, t_mp)
            
            # Calcular zeta(s)
            zeta_val = mp.zeta(s)
            
            # Calcular theta(t)
            theta_val = mp.mpf(self.compute_theta(float(t_mp)))
            
            # Z(t) = e^(i*theta) * zeta(1/2 + it)
            # Como queremos o valor real: Z(t) = |zeta| * cos(theta - arg(zeta))
            magnitude = mp.fabs(zeta_val)
            arg_zeta = mp.arg(zeta_val)
            
            Z = magnitude * mp.cos(theta_val - arg_zeta)
            
            result = float(Z)
            
            # Verificar se o resultado é válido
            if np.isnan(result) or np.isinf(result):
                print(f"Aviso: Z({t}) = {result}, usando aproximação")
                return self.compute_Z_riemann_siegel(t)
            
            return result
            
        except Exception as e:
            print(f"Erro ao calcular Z({t}): {e}, usando aproximação Riemann-Siegel")
            return self.compute_Z_riemann_siegel(t)
    
    def compute_Z_riemann_siegel(self, t):
        """
        Calcula Z(t) usando a fórmula de Riemann-Siegel (mais estável para t grandes)
        Z(t) = 2 * Σ[n=1 to N] cos(θ(t) - t*ln(n)) / sqrt(n)
        onde N = floor(sqrt(t/2π))
        """
        try:
            if t <= 0:
                return 0.0
            
            N = int(np.floor(np.sqrt(t / (2 * np.pi))))
            theta_t = self.compute_theta(t)
            
            Z = 0.0
            for n in range(1, min(N + 1, 1000)):  # Limitar para evitar loops muito longos
                term = 2 * np.cos(theta_t - t * np.log(n)) / np.sqrt(n)
                Z += term
            
            return Z
        except:
            return 0.0
    
    def compute_riemann_siegel_terms(self, t, n_terms=10):
        """
        Calcula os termos da fórmula de Riemann-Siegel (equação 17)
        Z(t) = 2 * Σ[n=1 to N] cos(θ(t) - t*ln(n)) / sqrt(n) + R
        onde N = floor(sqrt(t/2π))
        
        Retorna os primeiros n_terms termos
        """
        if t <= 0:
            return [0.0] * n_terms
        
        N = int(np.floor(np.sqrt(t / (2 * np.pi))))
        theta_t = self.compute_theta(t)
        
        terms = []
        for n in range(2, min(N + 1, n_terms + 2)):
            term = 2 * np.cos(theta_t - t * np.log(n)) / np.sqrt(n)
            terms.append(term)
        
        # Preencher com zeros se necessário
        while len(terms) < n_terms:
            terms.append(0.0)
        
        return terms[:n_terms]
    
    def load_or_download_zeros(self, use_precomputed=True):
        """
        Carrega zeros da função zeta a partir de arquivo local.
        Se não existir, faz download da base de Odlyzko.
        """
        zeros_file = "../dataset/riemann_zeros.txt"
        url = "https://www-users.cse.umn.edu/~odlyzko/zeta_tables/zeros1"

        if use_precomputed and os.path.exists(zeros_file):
            zeros_full = np.loadtxt(zeros_file)
            print(f"Carregados {len(zeros_full)} zeros do arquivo local")

        else:
            print("Arquivo não encontrado. Baixando zeros de Odlyzko...")
            response = requests.get(url)
            response.raise_for_status()

            zeros_full = np.fromstring(response.text, sep="\n")
            np.savetxt(zeros_file, zeros_full)

            print(f"Download concluído: {len(zeros_full)} zeros salvos em '{zeros_file}'")

        # Usar apenas os primeiros n_zeros
        if len(zeros_full) >= self.n_zeros:
            self.zeros = zeros_full[:self.n_zeros]
            print(f"Usando os primeiros {self.n_zeros} zeros")
        else:
            print(
                f"AVISO: apenas {len(zeros_full)} zeros disponíveis; "
                f"{self.n_zeros} foram solicitados"
            )
            self.zeros = zeros_full
            self.n_zeros = len(zeros_full)
    
    def compute_all_gram_points(self):
        """Calcula todos os pontos de Gram necessários"""
        print("Computando pontos de Gram...")
        self.gram_points = []
        
        # Computar exatamente n_zeros Gram points
        for n in range(1, self.n_zeros + 1):
            if n % 1000 == 0:
                print(f"Gram point {n}/{self.n_zeros}")
            
            gram = self.compute_gram_point(n)
            self.gram_points.append(gram)
        
        self.gram_points = np.array(self.gram_points)
        print(f"Total de Gram points: {len(self.gram_points)}")
    
    def compute_all_co_gram_points(self):
        """Calcula todos os co-Gram points necessários"""
        print("Computando co-Gram points...")
        self.co_gram_points = []
        
        # Computar exatamente n_zeros co-Gram points
        for n in range(1, self.n_zeros + 1):
            if n % 1000 == 0:
                print(f"Co-Gram point {n}/{self.n_zeros}")
            
            co_gram = self.compute_co_gram_point(n)
            self.co_gram_points.append(co_gram)
        
        self.co_gram_points = np.array(self.co_gram_points)
        print(f"Total de co-Gram points: {len(self.co_gram_points)}")
    
    def compute_distance(self):
        """
        Calcula a variável distance: γ_n - g_{n-1}
        onde γ_n é o n-ésimo zero e g_{n-1} é o (n-1)-ésimo ponto de Gram
        """
        print("Computando variável distance...")
        print(f"Tamanho de zeros: {len(self.zeros)}")
        print(f"Tamanho de gram_points: {len(self.gram_points)}")
        
        # Verificar se os tamanhos são compatíveis
        if len(self.zeros) != len(self.gram_points):
            raise ValueError(f"Tamanhos incompatíveis: zeros={len(self.zeros)}, gram_points={len(self.gram_points)}")
        
        # Inicializar array com tamanho correto
        n = len(self.zeros)
        self.distance = np.zeros(n)
        
        # Primeiro elemento (caso especial) - usa o próprio gram point
        self.distance[0] = self.zeros[0] - self.gram_points[0]
        
        # Demais elementos: zero[i] - gram_point[i-1]
        for i in range(1, n):
            self.distance[i] = self.zeros[i] - self.gram_points[i-1]
        
        print(f"Distance computado com sucesso. Tamanho: {len(self.distance)}")
    
    def create_lagged_features(self, series, n_lags):
        """
        Cria features com lags de uma série temporal
        
        Args:
            series: série temporal
            n_lags: número de lags a criar
        """
        lagged = np.zeros((len(series), n_lags))
        
        for lag in range(1, n_lags + 1):
            if lag < len(series):
                lagged[lag:, lag-1] = series[:-lag]
        
        return lagged
    
    def generate_all_features(self):
        """
        Gera todas as 94 features candidatas mencionadas no artigo
        """
        print("\n=== Gerando todas as features ===\n")
        
        # 1. Carregar ou computar zeros
        self.load_or_download_zeros()
        
        # 2. Computar Gram points
        self.compute_all_gram_points()
        
        # 3. Computar co-Gram points
        self.compute_all_co_gram_points()
        
        # 4. Computar distance (variável alvo)
        self.compute_distance()
        
        # 5. Criar DataFrame para armazenar features
        n_samples = len(self.zeros)
        features = {}
        
        # Feature 1: Gram points
        print("Adicionando Gram points...")
        features['gram'] = self.gram_points
        
        # Features 2-11: Lags de Gram points (1-10)
        print("Adicionando lags de Gram points...")
        gram_lags = self.create_lagged_features(self.gram_points, 10)
        for i in range(10):
            features[f'gram_lag_{i+1}'] = gram_lags[:, i]
        
        # Feature 12: co-Gram points
        print("Adicionando co-Gram points...")
        features['co_gram'] = self.co_gram_points
        
        # Features 13-22: Lags de co-Gram points (1-10)
        print("Adicionando lags de co-Gram points...")
        co_gram_lags = self.create_lagged_features(self.co_gram_points, 10)
        for i in range(10):
            features[f'co_gram_lag_{i+1}'] = co_gram_lags[:, i]
        
        # Features 23-32: Z-function em Gram points e seus lags
        print("Computando Z-function em Gram points (pode demorar)...")
        z_gram = []
        for idx, g in enumerate(self.gram_points):
            if idx % 1000 == 0:
                print(f"  Z(gram[{idx}])...")
            z_val = self.compute_Z_function(g)
            z_gram.append(z_val)
        z_gram = np.array(z_gram)
        features['z_gram'] = z_gram
        
        print(f"Z_gram - min: {z_gram.min():.4f}, max: {z_gram.max():.4f}, mean: {z_gram.mean():.4f}")
        
        z_gram_lags = self.create_lagged_features(z_gram, 10)
        for i in range(10):
            features[f'z_gram_lag_{i+1}'] = z_gram_lags[:, i]
        
        # Features 33-47: Z-function em co-Gram points e seus lags
        print("Computando Z-function em co-Gram points (pode demorar)...")
        z_co_gram = []
        for idx, cg in enumerate(self.co_gram_points):
            if idx % 1000 == 0:
                print(f"  Z(co_gram[{idx}])...")
            z_val = self.compute_Z_function(cg)
            z_co_gram.append(z_val)
        z_co_gram = np.array(z_co_gram)
        features['z_co_gram'] = z_co_gram
        
        print(f"Z_co_gram - min: {z_co_gram.min():.4f}, max: {z_co_gram.max():.4f}, mean: {z_co_gram.mean():.4f}")
        
        z_co_gram_lags = self.create_lagged_features(z_co_gram, 15)
        for i in range(15):
            features[f'z_co_gram_lag_{i+1}'] = z_co_gram_lags[:, i]
        
        # Features 48-57: Termos da fórmula de Riemann-Siegel (2-10)
        print("Computando termos de Riemann-Siegel...")
        for term_idx in range(2, 11):
            term_values = []
            for g in self.gram_points:
                terms = self.compute_riemann_siegel_terms(g, n_terms=10)
                term_values.append(terms[term_idx-2] if term_idx-2 < len(terms) else 0)
            features[f'z_term_{term_idx}'] = np.array(term_values)
        
        # Features 58-82: Lags de distance (1-25)
        print("Adicionando lags de distance...")
        distance_lags = self.create_lagged_features(self.distance, 25)
        for i in range(25):
            features[f'd_lag_{i+1}'] = distance_lags[:, i]
        
        # Features 83-93: Z-function em inteiros sucessivos e lags
        print("Computando Z-function em inteiros...")
        z_integers = []
        for n in range(1, n_samples + 1):
            if n % 1000 == 0:
                print(f"  Z({n})...")
            z_val = self.compute_Z_function(float(n))
            z_integers.append(z_val)
        z_integers = np.array(z_integers)
        features['z_integer'] = z_integers
        
        print(f"Z_integer - min: {z_integers.min():.4f}, max: {z_integers.max():.4f}, mean: {z_integers.mean():.4f}")
        
        z_int_lags = self.create_lagged_features(z_integers, 10)
        for i in range(10):
            features[f'z_integer_lag_{i+1}'] = z_int_lags[:, i]
        
        # Criar DataFrame
        df = pd.DataFrame(features)
        
        # Adicionar variável alvo
        df['distance'] = self.distance
        
        # Remover primeiras observações (edge effects)
        df = df.iloc[self.start_index:].reset_index(drop=True)
        
        print(f"\n=== Features geradas com sucesso! ===")
        print(f"Shape do dataset: {df.shape}")
        num_features = df.shape[1] - 1
        num_obs = df.shape[0]
        print(f"Número de features: {num_features}")
        print(f"Número de observações: {num_obs}")
        
        return df
    
    def save_features(self, df, filename='riemann_zeta_features.csv'):
        """Salva features em arquivo CSV"""
        df.to_csv(filename, index=False)
        print(f"\nFeatures salvas em: {filename}")
    
    def get_feature_statistics(self, df):
        """Retorna estatísticas das features"""
        stats = {
            'mean': df.mean(),
            'std': df.std(),
            'min': df.min(),
            'max': df.max(),
            'missing': df.isnull().sum()
        }
        
        return pd.DataFrame(stats)


# ============================================================================
# EXEMPLO DE USO
# ============================================================================

if __name__ == "__main__":
    
    # Configurações
    N_ZEROS = 90000  # Reduzido para teste mais rápido
    START_INDEX = 1001
    
    print("="*70)
    print("GERADOR DE FEATURES PARA ZEROS DA FUNÇÃO ZETA DE RIEMANN")
    print("="*70)
    
    # Criar gerador
    generator = RiemannZetaFeatureGenerator(
        n_zeros=N_ZEROS,
        start_index=START_INDEX
    )
    
    # Gerar todas as features
    df_features = generator.generate_all_features()
    
    # Mostrar primeiras linhas
    print("\n=== Primeiras 5 linhas do dataset ===")
    print(df_features.head())
    
    # Estatísticas
    # print("\n=== Estatísticas das features ===")
    # stats = generator.get_feature_statistics(df_features)
    # print(stats.head(20))
    
    # Salvar
    generator.save_features(df_features, '../dataset/riemann_features.csv')
    
    # Informações sobre a variável alvo (distance)
    # print("\n=== Estatísticas da variável alvo (distance) ===")
    # print(f"Média: {df_features['distance'].mean():.4f}")
    # print(f"Desvio padrão: {df_features['distance'].std():.4f}")
    # print(f"Mínimo: {df_features['distance'].min():.4f}")
    # print(f"Máximo: {df_features['distance'].max():.4f}")
    
    print("\n=== Processo concluído! ===")


GERADOR DE FEATURES PARA ZEROS DA FUNÇÃO ZETA DE RIEMANN

=== Gerando todas as features ===

Arquivo não encontrado. Baixando zeros de Odlyzko...
Download concluído: 100000 zeros salvos em '../dataset/riemann_zeros.txt'
Usando os primeiros 90000 zeros
Computando pontos de Gram...
Gram point 1000/90000
Gram point 2000/90000
Gram point 3000/90000
Gram point 4000/90000
Gram point 5000/90000
Gram point 6000/90000
Gram point 7000/90000
Gram point 8000/90000
Gram point 9000/90000
Gram point 10000/90000
Gram point 11000/90000
Gram point 12000/90000
Gram point 13000/90000
Gram point 14000/90000
Gram point 15000/90000
Gram point 16000/90000
Gram point 17000/90000
Gram point 18000/90000
Gram point 19000/90000
Gram point 20000/90000
Gram point 21000/90000
Gram point 22000/90000
Gram point 23000/90000
Gram point 24000/90000
Gram point 25000/90000
Gram point 26000/90000
Gram point 27000/90000
Gram point 28000/90000
Gram point 29000/90000
Gram point 30000/90000
Gram point 31000/90000
Gram point 3200

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


class RiemannFeatureImportanceAdvanced:
    """
    Implementa análise aprimorada de importância das variáveis
    baseada em mRMR + RandomForest + Rede Neural,
    similar ao método do paper “Predicting Zeros of the Riemann Zeta Function”.
    """

    def __init__(self, data_path='../dataset/riemann_features.csv', n_features_plot=40):
        self.data_path = data_path
        self.df = None
        self.X = None
        self.y = None
        self.feature_names = None
        self.scaler = MinMaxScaler()
        self.results = None
        self.n_features_plot = n_features_plot

    def load_and_prepare(self):
        """Carrega os dados e normaliza (MinMaxScaler como no artigo)."""
        print(f"\n{'='*70}")
        print("CARREGANDO E NORMALIZANDO OS DADOS")
        print(f"{'='*70}")

        self.df = pd.read_csv(self.data_path)
        if 'distance' not in self.df.columns:
            raise ValueError("Arquivo CSV precisa ter coluna 'distance' como variável alvo")

        # Separar alvo e variáveis
        self.y = self.df['distance'].values
        self.X = self.df.drop(columns=['distance']).values
        self.feature_names = self.df.drop(columns=['distance']).columns.tolist()

        print(f"Dataset carregado: {self.X.shape[0]} observações, {self.X.shape[1]} features.")
        
        # Normalizar entre 0–1, como no artigo
        self.X = self.scaler.fit_transform(self.X)
        print("Normalização MinMax aplicada a todas as variáveis.")

    def compute_linear_correlation(self):
        """Calcula relevância linear absoluta (|corr(feature, target)|)."""
        print("\nCalculando correlações lineares...")
        corr_scores = []
        for i in range(self.X.shape[1]):
            corr, _ = pearsonr(self.X[:, i], self.y)
            corr_scores.append(abs(corr))
        return np.array(corr_scores)

    def compute_mutual_info(self):
        """Calcula Mutual Information entre cada feature e o target."""
        print("Calculando Mutual Information (não-linear)...")
        mi_scores = mutual_info_regression(self.X, self.y, random_state=42)
        mi_scores /= mi_scores.max()  # normalizar 0–1
        return mi_scores

    def compute_random_forest_importance(self, n_trees=300):
        """Usa RandomForestRegressor para estimar feature importances."""
        print("Treinando RandomForest para avaliar importância conjunta...")
        rf = RandomForestRegressor(
            n_estimators=n_trees,
            max_depth=None,
            min_samples_split=4,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(self.X, self.y)
        rf_scores = rf.feature_importances_
        rf_scores /= rf_scores.max()
        return rf_scores

    def compute_neural_importance(self, hidden=(50, 30), max_iter=400):
        """Calcula importância pseudo-“à la Gevrey” (ANN weights)."""
        print("Treinando MLPRegressor (estimando pesos de entrada)...")
        mlp = MLPRegressor(hidden_layer_sizes=hidden, 
                            activation='relu', 
                            solver='adam', 
                            random_state=42, 
                            max_iter=max_iter)
        mlp.fit(self.X, self.y)
        y_pred = mlp.predict(self.X)
        r2 = r2_score(self.y, y_pred)
        print(f"R² no treino (indicativo): {r2:.3f}")

        # Soma absoluta dos pesos de entrada (camada 1)
        W = np.abs(mlp.coefs_[0])
        mean_weight = W.sum(axis=1)
        mean_weight /= mean_weight.max()
        return mean_weight

    def combine_scores(self, corr, mi, rf, nn=None):
        """Combina scores distintos em uma métrica mRMR-like."""
        print("Combinando as métricas em um score final (mRMR híbrido)...")

        combined = 0.4 * corr + 0.3 * mi + 0.3 * rf
        if nn is not None:
            combined = (combined + nn) / 2.0
        return combined / combined.max()

    def run_analysis(self, include_nn=True):
        """Executa toda a análise."""
        self.load_and_prepare()
        corr = self.compute_linear_correlation()
        mi = self.compute_mutual_info()
        rf = self.compute_random_forest_importance()
        nn = None
        if include_nn:
            nn = self.compute_neural_importance()

        combined = self.combine_scores(corr, mi, rf, nn)

        # Criar DataFrame com resultados
        df_imp = pd.DataFrame({
            'feature': self.feature_names,
            'corr': corr,
            'mutual_info': mi,
            'rf_importance': rf,
            'nn_weight': nn if nn is not None else 0,
            'final_score': combined
        }).sort_values('final_score', ascending=False)

        self.results = df_imp.reset_index(drop=True)
        print("\nAnálise concluída. Top 10 variáveis:")
        print(self.results.head(10).to_string(index=False))
        return self.results

    def plot_importance(self, filename='feature_importance_mrmr.png'):
        """Plota gráfico similar ao da Figura 4 do artigo."""
        if self.results is None:
            raise RuntimeError("Execute run_analysis() primeiro.")

        top_df = self.results.head(self.n_features_plot)
        plt.figure(figsize=(10, 12))
        sns.barplot(
            data=top_df, 
            x='final_score', 
            y='feature', 
            color='crimson'
        )
        plt.title(f"Top {self.n_features_plot} Features Mais Importantes (mRMR + RF + NN)")
        plt.xlabel("Score de Importância Combinado (normalizado)")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.savefig(filename, dpi=300)
        print(f"\nGráfico salvo: {filename}")
        plt.close()


# ===================================================================
# USO TÍPICO:
# ===================================================================

if __name__ == "__main__":
    print("="*70)
    print("ANÁLISE APRIMORADA DE IMPORTÂNCIA DAS FEATURES (mRMR + RF + NN)")
    print("="*70)

    analyzer = RiemannFeatureImportanceAdvanced(data_path='../dataset/riemann_features.csv')
    results = analyzer.run_analysis(include_nn=True)
    analyzer.plot_importance()

    # Salvar CSV com importância
    results.to_csv("../results/feature_importance_mrmr_results.csv", index=False)
    print("\nResultados salvos em '../results/feature_importance_mrmr_results.csv'")

ANÁLISE APRIMORADA DE IMPORTÂNCIA DAS FEATURES (mRMR + RF + NN)

CARREGANDO E NORMALIZANDO OS DADOS
Dataset carregado: 88999 observações, 94 features.
Normalização MinMax aplicada a todas as variáveis.

Calculando correlações lineares...
Calculando Mutual Information (não-linear)...
Treinando RandomForest para avaliar importância conjunta...
Treinando MLPRegressor (estimando pesos de entrada)...
R² no treino (indicativo): 0.981
Combinando as métricas em um score final (mRMR híbrido)...

Análise concluída. Top 10 variáveis:
        feature     corr  mutual_info  rf_importance  nn_weight  final_score
z_co_gram_lag_2 0.015175     1.000000       1.000000   1.000000     1.000000
        d_lag_1 0.183591     0.078943       0.397279   0.901641     0.696074
   z_gram_lag_1 0.004842     0.567796       0.846151   0.684523     0.691529
         z_gram 0.012653     0.425011       0.417292   0.783022     0.648025
z_co_gram_lag_1 0.019686     0.222815       0.010426   0.807698     0.551374
        d